<a href="https://colab.research.google.com/github/paulinaeb/IDaSec-project/blob/exp4/Evasion_notebooks/textfooler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TextFooler Adversarial Attack on Spam Classifier

This notebook demonstrates the use of the TextFooler attack to test the robustness of a spam classifier.
TextFooler replaces important words with semantically similar alternatives to trick a model into misclassification while preserving human readability.

## Step 1: Install Dependencies

We install the required libraries:  
- `textattack` for adversarial attacks  
- `transformers` for loading pre-trained models

In [2]:
# STEP 1: Install dependencies
!pip install textattack transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.7/54.7 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 42.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 62.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 445.7/445.7 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Step 2: Import Libraries

Import modules from `textattack`, `transformers`, and other support libraries to construct and run the attack.

In [3]:
# STEP 2: Import libraries
from textattack.attack_recipes import TextFoolerJin2019
from textattack.models.wrappers import HuggingFaceModelWrapper
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from textattack.datasets import Dataset
from textattack import Attacker, AttackArgs
import pandas as pd
import nltk

textattack: Updating TextAttack package dependencies.
textattack: Downloading NLTK required packages.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package omw to /root/nltk_data...
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


## Step 3: Load Dataset

Load the Enron email dataset (training split). Adjust the file path as needed.

In [4]:
# STEP 3: Load the Enron dataset
df = pd.read_csv("enron1_train.csv")

## Step 4: Select Example Spam Messages

We select a few spam messages to run through the TextFooler attack. These are labeled as class `1` (spam).

In [5]:
# STEP 4: Sample a few spam messages
sample_spam = df[df["target"] == "spam"]["email"].head(5)
examples = [(text, 1) for text in sample_spam.tolist()]  # Label 1 = spam
dataset = Dataset(examples)

## Step 5: Load Spam Classifier

We use a small pre-trained BERT model fine-tuned for SMS spam detection, wrapped for compatibility with TextAttack.

In [6]:
# STEP 5: Load spam classifier model (you can replace with your own)
model_name = "mrm8488/bert-tiny-finetuned-sms-spam-detection"
model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_wrapper = HuggingFaceModelWrapper(model, tokenizer)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/17.6M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/324 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## Step 6: Initialize TextFooler Attack

We use TextAttack’s preconfigured implementation of TextFooler.

In [7]:
# STEP 6: Build the TextFooler attack
attack = TextFoolerJin2019.build(model_wrapper)

textattack: Downloading https://textattack.s3.amazonaws.com/word_embeddings/paragramcf.
100%|██████████| 481M/481M [00:12<00:00, 39.8MB/s]
textattack: Unzipping file /root/.cache/textattack/tmp0_bt1nes.zip to /root/.cache/textattack/word_embeddings/paragramcf.
textattack: Successfully saved word_embeddings/paragramcf to cache.
textattack: Unknown if model of class <class 'transformers.models.bert.modeling_bert.BertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


## Download NLTK Resources (If Missing)

These are needed for part-of-speech tagging in TextFooler. You may skip this if already installed.

In [8]:
nltk.download('averaged_perceptron_tagger') # Download the missing resource

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [9]:
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

## Step 7: Run the Attack

Run the TextFooler attack against the selected spam samples. Results will show original vs. adversarial text and model predictions.

In [10]:
# STEP 7: Set up and run the attack
attack_args = AttackArgs(
    num_examples=len(examples),  # Attack all provided samples
    disable_stdout=False,        # Set to True to suppress output
)

attacker = Attacker(attack, dataset, attack_args)
attacker.attack_dataset()

Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  delete
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapEmbedding(
    (max_candidates):  50
    (embedding):  WordEmbedding
  )
  (constraints): 
    (0): WordEmbeddingDistance(
        (embedding):  WordEmbedding
        (min_cos_sim):  0.5
        (cased):  False
        (include_unknown_words):  True
        (compare_against_original):  True
      )
    (1): PartOfSpeech(
        (tagger_type):  nltk
        (tagset):  universal
        (allow_verb_noun_swap):  True
        (compare_against_original):  True
      )
    (2): UniversalSentenceEncoder(
        (metric):  angular
        (threshold):  0.840845057
        (window_size):  15
        (skip_text_shorter_than_window):  True
        (compare_against_original):  False
      )
    (3): RepeatModification
    (4): StopwordModification
    (5): InputColumnModification(
        (matching_column_labels):  ['premise', 'hypothesis']
       

 20%|██        | 1/5 [00:23<01:32, 23.11s/it]

--------------------------------------------- Result 1 ---------------------------------------------


[Succeeded / Failed / Skipped / Total] 1 / 0 / 0 / 1:  20%|██        | 1/5 [00:23<01:33, 23.40s/it]

[[1 (80%)]] --> [[0 (50%)]]

Subject: hi agaain [[hello]] , [[welcome]] to pharm laryngology [[online]] sh soundless op - one of the rocketeer leading oniine [[pharmaceutical]] [[shops]] cumulation [[v]] affixation g a [[solicitor]] l fictile ll l [[montage]] a [[r]] assess ac nautical l i medusa sv golliwog a reeded um andmanyother . - [[save]] over 5 durable 0 % - [[worldwide]] shlppln airspraying [[g]] - [[total]] confidentiai inheritor ity - over 5 miiiion custome roland [[rs]] in 130 [[countries]] have a nice d impendent ay !

Subject: hi agaain [[heh]] , [[happy]] to pharm laryngology [[electronic]] sh soundless op - one of the rocketeer leading oniine [[medicinal]] [[grocers]] cumulation [[against]] affixation g a [[counsels]] l fictile ll l [[fixation]] a [[p]] assess ac nautical l i medusa sv golliwog a reeded um andmanyother . - [[spared]] over 5 durable 0 % - [[generalized]] shlppln airspraying [[ke]] - [[utterly]] confidentiai inheritor ity - over 5 miiiion custome roland [

...

--------------------------------------------- Result 2 ---------------------------------------------
[[1 (74%)]] --> [[0 (52%)]]

[[Subject]]: instructions to remove spyware / adware [[infections]] this [[message]] [[will]] [[inform]] you on how to remove spyware and adware from your [[pcs]] hard [[drive]] . please read below carefully : if you , or someone else that [[uses]] your [[pc]] , have been downloading [[internet]] [[files]] such as [[music]] , [[games]] , or [[movies]] , then adware and spyware [[programs]] may have been added to your computer ' s hard drive without your [[direct]] knowledge . to check for any adware or spyware [[applications]] press here . there is no cost for this scan : if after completing the complimentary [[scan]] it is brought to your attention that your computer ' s hard [[drive]] is [[infected]] with adware , spyware , or both , then it may be in your computer ' s best interest to remove the adware and spyware [[applications]] . [[click]] here to [[be

[Succeeded / Failed / Skipped / Total] 2 / 1 / 0 / 3:  60%|██████    | 3/5 [01:44<01:09, 34.91s/it]

--------------------------------------------- Result 3 ---------------------------------------------
[[1 (87%)]] --> [[[FAILED]]]

Subject: acrobat pro 7 . 0 $ 69 . 95 xp pro opt - in email special offer unsubscribe me search software top 10 new titles on sale now ! 1 office pro 20032 windows xp pro 3 adobe creative suite premium 4 norton antivirus 20055 flash mx 20046 corel draw 127 adobe acrobat 7 . 08 windows 2003 server 9 alias maya 6 wavefrtl 0 adobe premiere see more by this manufacturer microsoft apple software customers also bought these other items . . . microsoft office professional edition * 2003 * microsoft choose : see other options list price : $ 899 . 00 price : $ 69 . 99 you save : $ 830 . 01 ( 92 % ) availability : available for instant download ! coupon code : ise 229 media : cd - rom / download system requirements | accessories | other versionsfeatures : analyze and manage business information using access databases exchange data with other systems using enhanced xml

[Succeeded / Failed / Skipped / Total] 2 / 2 / 1 / 5: 100%|██████████| 5/5 [01:56<00:00, 23.39s/it]

--------------------------------------------- Result 4 ---------------------------------------------
[[1 (89%)]] --> [[[FAILED]]]

Subject: 2 x - message - info : s 63 h 5 ja 567 vssd 2 p 26 n 61 wl 67 hzvlnqxjn received : from mimbdzl 7 . hinet . net ( [ 188 . 104 . 212 . 29 ] ) by b 255 - jyx . hinet . net with microsoft smtpsvc ( 5 . 0 . 2195 . 6824 ) ; tue , 23 mar 2004 00 : 38 : 43 - 0100 received : from characteristicwzo 9 ( damsel [ 94 . 236 . 64 . 48 ] ) by hinet . net ( dr 937 ) with smtp id 619255 c 738 ox ( authid : mitchelforeman ) ; mon , 22 mar 2004 23 : 37 : 43 - 0200 from : corine browning xlvjagegpeudkw @ wanadoo . fr to : ' paliourg ' paliourg @ iit . demokritos . gr 2 subject : [ spam ? ] fwd : doctors not needed . secure shopping . vcodln . v | @ gra . xan @ x . valii | um . twjticwy date : mon , 22 mar 2004 19 : 39 : 43 - 0600 message - id : mime - version : 1 . 0 content - type : multipart / alternative ; boundary = - - 2080774744588021497 - - - - 2080774744588021